# Structural Signal Validation – secfsn_plus_structural

This notebook validates the structural scoring model built in `secfsn_plus_structural` using real SEC data.

**Pipeline**

SEC → canonical ratios → structural diagnostics → scoring

**Goal**

Validate signal quality and interpretability through distribution checks, diagnostics, and a latest-period top/bottom comparison.

## 2. Imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent
src_path = repo_root / "src"

if not src_path.exists():
    repo_root = Path.cwd()
    src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## 3. Load Pipeline

In [ ]:
from secfsn_plus import build_feature_panel
from secfsn_plus_structural import build_structural_features

## 4. Configuration

In [ ]:
PERIODS = [(2023, 4), (2024, 1), (2024, 2)]

## 5. Build Dataset

In [ ]:
panel = build_feature_panel(periods=PERIODS)
features = build_structural_features(panel)

if "period" not in features.columns and "period" in getattr(features.index, "names", []):
    features = features.copy()
    features["period"] = features.index.get_level_values("period")

print("Input columns:", len(panel.columns))
print("Output columns:", len(features.columns))
print("Rows:", len(features))

## 6. Diagnostics

In [ ]:
diagnostics = features.attrs.get("diagnostics", {})
print(diagnostics)

## 7. Score Distribution (Per Period)

In [ ]:
features.groupby("period")["structural_score"].describe()

## 8. Latest Snapshot Analysis

In [ ]:
latest = features.index.get_level_values("period").max()
df_latest = features.xs(latest, level="period")

top = df_latest.sort_values("structural_rank").tail(10)
bottom = df_latest.sort_values("structural_rank").head(10)

cols = [
    "structural_score",
    "accrual_intensity",
    "non_cash_earnings_ratio",
    "roe_stability_8q",
    "margin_volatility",
    "signal_decay_autocorr_12q",
]

print(f"Latest period: {latest}")
print("Top 10 structural rank")
top[cols]

In [ ]:
print("Bottom 10 structural rank")
bottom[cols]

## 9. Interpretation Section

A **high `structural_score`** indicates a company whose accounting and operating profile looks more structurally robust under the model: lower accrual pressure, more cash-backed earnings, and more stable profitability behavior over time.

A **low `structural_score`** points to weaker structural quality signals: elevated accrual intensity, less reliable earnings composition, and less stable margins or returns on equity. These firms may deserve deeper forensic review before being treated as high-confidence candidates.

**Accruals and stability matter together** because they capture different dimensions of quality. Accrual-heavy earnings can be more sensitive to estimation choices or transient working-capital moves, while stability measures help distinguish durable operating performance from noisy or deteriorating fundamentals.

## 10. Distribution Plot

In [ ]:
features["structural_rank"].hist(bins=50)
plt.title("Structural Rank Distribution")
plt.xlabel("Structural Rank")
plt.ylabel("Frequency")
plt.show()

## 11. Optional Correlation Check

In [ ]:
cols = [
    "structural_score",
    "accrual_intensity",
    "non_cash_earnings_ratio",
    "roe_stability_8q",
    "margin_volatility",
]

features[cols].corr()

## 12. Conclusion

This validation notebook is designed to provide a fast qualitative check on **signal validity** by showing whether the structural score produces a sensible distribution, interpretable diagnostics, and a clear separation between the strongest and weakest latest-period observations.

**Strengths** include transparent decomposition into accounting-quality and stability-oriented components, period-level summary statistics, and an immediately reviewable top/bottom snapshot for demos or blog-style walkthroughs.

**Limitations** include dependence on upstream feature availability, sensitivity to data sparsity in real SEC histories, and the fact that this notebook validates interpretability more than realized predictive performance.

**Next steps** are to connect the structural score to downstream modeling or trading evaluation, such as return spreads, hit-rate tests, ranking stability, and portfolio construction diagnostics.